In [ ]:
import sys, yaml, json
from pathlib import Path
from IPython.display import display, Markdown

# Read the config file
config_path = Path.home() / ".abf2nwb_config.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

subfolder = "L1.ST8"
username = config["username"]

path_analysis = config["path_analysis"]

root = (
    Path("/Users") / config["username"] / "Library"
    / Path(config["path_analysis"]).relative_to("/Library")
    / "NWB data" / "Figure 11 RNAscope" / subfolder
)

nwb_path = root / f"{subfolder}.nwb"

# add required functions
repo_root = Path("/Users") / config["username"] / "Documents" / "Repositories" / "CatalystNeuro"
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# from rnascope_functions import *

from master_RNAscope import *



In [ ]:
from pathlib import Path
import pandas as pd

analysis_dir = root / "analysis"
json_paths = sorted(analysis_dir.glob("*.roi_analysis.json"))

new_params = {
    "detection_method": "fiji",
    "sigma_small": 1.0,
    "sigma_large": 2.8,
    "threshold_percentile": 99,
    "peak_footprint": 4,
    "maxima_tolerance": 160,
    "show_detected": False,
    "show_verify": False,  
}

all_results = []
all_states = {}
all_payloads = {}

reanalysis_dir = root / "reanalysis"
reanalysis_dir.mkdir(exist_ok=True)

for json_path in json_paths:
    state, analysis_payload = reconstruct_state_from_saved_analysis(
        nwb_path,
        analysis_path=json_path,
        display_mode= "rendered_from_raw",  # "rendered_from_raw" or "exported_channel_tiffs"
    )

    results2, state = RNAscopeAnalysisFinish(state, **new_params)

    save_rnascope_field_analysis(
        state,
        results2,
        analysis_params=new_params,
        out_dir=reanalysis_dir,
    )

In [ ]:
from pathlib import Path
import json
import re
import pandas as pd

subfolder = "L1.ST8"

root = (
    Path("/Users") / config["username"] / "Library"
    / Path(config["path_analysis"]).relative_to("/Library")
    / "NWB data" / "Figure 11 RNAscope" / subfolder
)

analysis_dir = root / "reanalysis"
json_paths = sorted(analysis_dir.glob("*.roi_analysis.json"))

rows = []
for json_path in json_paths:
    payload = json.loads(json_path.read_text())
    field_name = str(payload["field"])

    m = re.match(r"^(?P<slice_id>.+)\.(?P<hemisphere>UL|L)_60x\.(?P<field_index>\d+)$", field_name)
    if m is None:
        raise ValueError(f"Could not parse field name: {field_name}")

    slice_id = m.group("slice_id")
    hemisphere = m.group("hemisphere")
    field_index = int(m.group("field_index"))
    condition = "Intact" if hemisphere == "UL" else "Lesioned"

    for group, rois in payload.get("groups", {}).items():
        for roi in rois:
            rows.append(
                {
                    "condition": condition,
                    "cell_type": str(group),
                    "slice_id": slice_id,
                    "field": field_name,
                    "hemisphere": hemisphere,
                    "field_index": field_index,
                    "replicate": int(roi["roi_index"]),
                    "count": int(roi.get("dot_count", 0)),
                }
            )

json_df = pd.DataFrame(rows, columns=[
    "condition",
    "cell_type",
    "slice_id",
    "field",
    "hemisphere",
    "field_index",
    "replicate",
    "count",
]).sort_values(
    ["cell_type", "condition", "field_index", "field", "replicate"],
    kind="stable",
).reset_index(drop=True)

json_df["field_index"] = json_df["field_index"].astype(str)

json_df

In [ ]:
with NWBHDF5IO(str(nwb_path), "r", load_namespaces=True) as io:
    nwbfile = io.read()
    imagej_df = (
        nwbfile.processing["rnascope_analysis_metadata"]["imagej_chrnb2_original_counts"]
        .to_dataframe()
        .reset_index(drop=True)
    )

imagej_df

In [ ]:
imagej_field = (
    imagej_df
    .groupby(
        ["condition", "cell_type", "slice_id", "field", "hemisphere", "field_index"],
        as_index=False,
    )["count"]
    .mean()
    .rename(columns={"count": "imagej_mean"})
)

analysis_field = (
    json_df
    .groupby(
        ["condition", "cell_type", "slice_id", "field", "hemisphere", "field_index"],
        as_index=False,
    )["count"]
    .mean()
    .rename(columns={"count": "analysis_mean"})
)

compare_field = imagej_field.merge(
    analysis_field,
    on=["condition", "cell_type", "slice_id", "field", "hemisphere", "field_index"],
    how="inner",
    validate="one_to_one",
)

pearson_r = compare_field["imagej_mean"].corr(compare_field["analysis_mean"], method="pearson")
spearman_r = compare_field["imagej_mean"].corr(compare_field["analysis_mean"], method="spearman")

print("n matched fields:", len(compare_field))
print("Pearson r:", round(pearson_r, 4))
print("Spearman r:", round(spearman_r, 4))

compare_field
